# exp_08 CHM Ablation

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

In [ ]:
config = load_experiment_config()
variants = config['setup_ablation']['chm']['variants']

train_df, val_df, _ = data_loading.load_berlin_splits(DATA_DIR / 'phase_2_splits')

results = []
for variant in variants:
    df_train = train_df.copy()
    df_val = val_df.copy()

    df_train = ablation.apply_chm_strategy(df_train, variant['name'])
    df_val = ablation.apply_chm_strategy(df_val, variant['name'])

    feature_cols = data_loading.get_feature_columns(
        df_train,
        include_chm=variant['name'] != 'no_chm',
        chm_features=variant['features'] or None,
    )
    x_train = df_train[feature_cols]
    x_val = df_val[feature_cols]

    y_train = df_train['genus_latin']
    y_val = df_val['genus_latin']
    y_train_enc, label_to_idx, _ = preprocessing.encode_genus_labels(y_train)
    y_val_enc = y_val.map(label_to_idx).to_numpy()

    x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(x_train, x_val=x_val)
    cv = training.create_spatial_block_cv(df_train, n_splits=config['global']['cv_folds'])

    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(rf, x_train_scaled, y_train_enc, df_train['block_id'].values, cv)

    results.append({
        'variant': variant['name'],
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_val_gap': metrics['train_val_gap'],
    })

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_08_chm_ablation'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_ablation_comparison(
    results_df,
    title='CHM Strategy Comparison',
    output_path=fig_dir / 'chm_ablation.png',
)

print('CHM ablation results:')
print(results_df.to_string(index=False))

In [ ]:
# Per-feature decision logic
rules = config['setup_ablation']['chm']['decision_rules']

# Train RF with all CHM features to get importances
all_chm_df_train = train_df.copy()
all_chm_features = data_loading.get_feature_columns(
    all_chm_df_train,
    include_chm=True,
    chm_features=['CHM_1m', 'CHM_1m_zscore', 'CHM_1m_percentile'],
)

x_all = all_chm_df_train[all_chm_features].to_numpy(dtype=float)
y_all, _, _ = preprocessing.encode_genus_labels(all_chm_df_train['genus_latin'])

# Compute feature importance
importance_df = ablation.compute_feature_importance(x_all, y_all, all_chm_features)
chm_importance = {row['feature']: row['importance'] for _, row in importance_df.iterrows()}

# Get baseline metrics (no_chm)
baseline_f1 = results_df.loc[results_df['variant'] == 'no_chm', 'val_f1_mean'].iloc[0]
baseline_gap = results_df.loc[results_df['variant'] == 'no_chm', 'train_val_gap'].iloc[0]

# Per-feature evaluation against 3 thresholds
included_features = []
per_feature_results = []

for feature_name, variant_name in [
    ('CHM_1m', 'raw_chm'),
    ('CHM_1m_zscore', 'zscore_only'),
    ('CHM_1m_percentile', 'percentile_only'),
]:
    row = results_df.loc[results_df['variant'] == variant_name].iloc[0]
    importance = chm_importance.get(feature_name, 0.0)
    gap_increase = row['train_val_gap'] - baseline_gap
    f1_improvement = row['val_f1_mean'] - baseline_f1

    excluded = False
    reason = ''

    # Threshold 1: Feature dominance check
    if importance > rules['importance_threshold']:
        excluded = True
        reason = f"Dominates model (importance={importance:.3f} > {rules['importance_threshold']})"
    # Threshold 2: Generalization destabilization check
    elif gap_increase > rules['max_gap_increase']:
        excluded = True
        reason = f"Destabilizes generalization (gap increase={gap_increase:.3f} > {rules['max_gap_increase']})"
    # Threshold 3: Marginal improvement check
    elif f1_improvement < rules['min_improvement']:
        excluded = True
        reason = f"Marginal improvement ({f1_improvement:.3f} < {rules['min_improvement']})"
    else:
        included_features.append(feature_name)
        reason = f"Passes all criteria (imp={importance:.3f}, gap_inc={gap_increase:.3f}, f1_imp={f1_improvement:.3f})"

    per_feature_results.append({
        'feature': feature_name,
        'variant': variant_name,
        'importance': float(importance),
        'gap_increase': float(gap_increase),
        'f1_improvement': float(f1_improvement),
        'included': not excluded,
        'reason': reason,
    })

# Map included features to variant name
variant_map = {
    (): 'no_chm',
    ('CHM_1m_zscore',): 'zscore_only',
    ('CHM_1m_percentile',): 'percentile_only',
    ('CHM_1m_zscore', 'CHM_1m_percentile'): 'both_engineered',
    ('CHM_1m',): 'raw_chm',
}
decision = variant_map.get(tuple(sorted(included_features)), 'no_chm')

print(f"\nPer-feature evaluation:")
for result in per_feature_results:
    status = 'INCLUDED' if result['included'] else 'EXCLUDED'
    print(f"  {result['feature']:20s} {status:8s} - {result['reason']}")

print(f"\nFinal decision: {decision}")
print(f"Included features: {included_features or 'none (Sentinel-2 only)'}")

In [ ]:
# Write decision to setup_decisions.json
setup_path = OUTPUT_DIR / 'metadata/setup_decisions.json'

setup = {
    'chm_strategy': {
        'decision': decision,
        'included_features': included_features,
        'reasoning': f"Per-feature evaluation: {len(included_features)} features passed thresholds",
        'per_feature_results': per_feature_results,
        'ablation_results': results_df.to_dict(orient='records'),
    }
}

setup_path.write_text(json.dumps(setup, indent=2))
print(f"\nWrote CHM decision to {setup_path}")

# Validate against schema
validate_setup_decisions(setup_path)
print("Schema validation: PASSED")